In [1]:
import os
import hashlib
import glob
import shutil
import subprocess
import re
import logging
import urllib.request
from urllib.error import URLError, HTTPError

# Configure the logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("dependency_download.log"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

# Define the target directory for dependencies
LIB_DIR = 'defects4j/framework/projects/lib'  # Replace with your actual path

# List of dependencies to download
dependencies = [
    {
        'group_id': 'junit',
        'artifact_id': 'junit',
        'version': '4.13.2',
    },
    {
        'group_id': 'org.mockito',
        'artifact_id': 'mockito-core',
        'version': '5.3.1',
    },
    {
        'group_id': 'org.powermock',
        'artifact_id': 'powermock-api-mockito2',
        'version': '2.0.9',
    },
    {
        'group_id': 'org.powermock',
        'artifact_id': 'powermock-module-junit4',
        'version': '2.0.9',
    },
    {
        'group_id': 'org.hamcrest',
        'artifact_id': 'hamcrest-core',
        'version': '2.1',
    },
    # Additional PowerMock dependencies
    {
        'group_id': 'org.powermock',
        'artifact_id': 'powermock-api-support',
        'version': '2.0.9',
    },
    {
        'group_id': 'org.powermock',
        'artifact_id': 'powermock-reflect',
        'version': '2.0.9',
    },
    {
        'group_id': 'org.objenesis',
        'artifact_id': 'objenesis',
        'version': '3.2',
    },
    # Adding Rhino dependency
    {
        'group_id': 'org.mozilla',
        'artifact_id': 'rhino',
        'version': '1.7.14',
    },
    # Add more dependencies here if necessary
]

def construct_maven_central_url(group_id, artifact_id, version):
    """
    Constructs the download URL for a Maven Central artifact.
    """
    base_url = 'https://repo1.maven.org/maven2/'
    group_path = group_id.replace('.', '/')
    return f"{base_url}{group_path}/{artifact_id}/{version}/{artifact_id}-{version}.jar"

def download_jar(url, dest_path):
    """
    Downloads a JAR file from the given URL to the destination path.
    """
    try:
        logger.info(f"Starting download: {url}")
        with urllib.request.urlopen(url) as response, open(dest_path, 'wb') as out_file:
            shutil.copyfileobj(response, out_file)
        logger.info(f"Successfully downloaded: {dest_path}")
    except HTTPError as e:
        logger.error(f"HTTP Error: {e.code} while downloading {url}")
    except URLError as e:
        logger.error(f"URL Error: {e.reason} while downloading {url}")
    except Exception as e:
        logger.error(f"Unexpected error while downloading {url}: {e}")

def ensure_lib_directory(lib_dir):
    """
    Ensures that the library directory exists.
    """
    if not os.path.exists(lib_dir):
        try:
            os.makedirs(lib_dir)
            logger.info(f"Created library directory: {lib_dir}")
        except Exception as e:
            logger.error(f"Failed to create library directory {lib_dir}: {e}")
            raise e
    else:
        logger.info(f"Library directory already exists: {lib_dir}")

def main():
    # Ensure the target library directory exists
    ensure_lib_directory(LIB_DIR)

    for dep in dependencies:
        group_id = dep['group_id']
        artifact_id = dep['artifact_id']
        version = dep['version']

        jar_name = f"{artifact_id}-{version}.jar"
        dest_path = os.path.join(LIB_DIR, jar_name)

        # Check if the JAR already exists
        if os.path.exists(dest_path):
            logger.info(f"JAR already exists, skipping download: {dest_path}")
            continue

        # Construct the download URL
        url = construct_maven_central_url(group_id, artifact_id, version)

        # Download the JAR
        download_jar(url, dest_path)

    logger.info("All dependencies have been processed.")

if __name__ == "__main__":
    main()


2025-01-14 20:07:28,578 - INFO - Library directory already exists: defects4j/framework/projects/lib
2025-01-14 20:07:28,579 - INFO - Starting download: https://repo1.maven.org/maven2/junit/junit/4.13.2/junit-4.13.2.jar
2025-01-14 20:07:28,661 - INFO - Successfully downloaded: defects4j/framework/projects/lib/junit-4.13.2.jar
2025-01-14 20:07:28,661 - INFO - Starting download: https://repo1.maven.org/maven2/org/mockito/mockito-core/5.3.1/mockito-core-5.3.1.jar
2025-01-14 20:07:28,736 - INFO - Successfully downloaded: defects4j/framework/projects/lib/mockito-core-5.3.1.jar
2025-01-14 20:07:28,737 - INFO - Starting download: https://repo1.maven.org/maven2/org/powermock/powermock-api-mockito2/2.0.9/powermock-api-mockito2-2.0.9.jar
2025-01-14 20:07:28,758 - INFO - Successfully downloaded: defects4j/framework/projects/lib/powermock-api-mockito2-2.0.9.jar
2025-01-14 20:07:28,758 - INFO - Starting download: https://repo1.maven.org/maven2/org/powermock/powermock-module-junit4/2.0.9/powermock-mo